### Goal
Apply the grid search over the LSTM and obtains the best hyperparameters

#### 1. Setup

In [ ]:
# Importing packages
import json
import torch
import pandas as pd
from pathlib import Path
from pa_model import Dataset, DATA_RAW, DATA_PROCESSED, RESULTS
from lstm import GridSearchLSTM

In [2]:
# Loading Dataset
path_initial_data = f"{DATA_RAW}/dadosIniciais.csv"
path_experiment_data = f"{DATA_PROCESSED}/expData.csv"

if Path(path_experiment_data).exists():
    ds = Dataset.from_csv(path_experiment_data)
else:
    df = pd.read_csv(path_initial_data)
    df = df.loc[df.index.repeat(10)].reset_index(drop=True)
    df = df.sample(frac=1,random_state=42).reset_index(drop=True)
    df.to_csv(path_experiment_data,index=False)
    ds = Dataset.from_csv(path_experiment_data)

In [ ]:
# Splitting Data into train, val and test Datasets
train, val, test = ds.split()

In [ ]:
 # Defining the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)

In [ ]:
# GridSearch Hyperparameters# Parametros do gridsearch
param_grid = {
    "window_size": [10, 20, 30],
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2, 3, 4, 5],
    "learning_rate": [0.001],
    "dropout": [0.0, 0.5],
    "batch_size": [128, 512, 1024],
}

#### 2. Running Grid Search

In [ ]:
# Creating LSTM Grid Search Object
gsearch = GridSearchLSTM(train, val, param_grid, device)

In [ ]:
# Running Grid Search
best_metric, best_model, best_params = gsearch.run(RESULTS)

In [ ]:
# Saving Results
dict_results = {"best_evm": best_metric}
dict_results.update(best_params)

with open(RESULTS/"best_result.json", "w") as f:
    json.dump(dict_results, f)

In [ ]:
# Saving Model
torch.save(best_model.state_dict(), RESULTS/"best_model.pkl")